In [3]:
# Load environment variables from a .env file
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# Create the OpenAI client. It automatically reads OPENAI_API_KEY
# from the environment we loaded above.
from openai import OpenAI
openai_client = OpenAI()

In [5]:
# Helper: send a prompt to the model and return just the text answer.
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini", # model used for generation
        input=prompt
    )
    return response.output_text # pull out the plain-text response

In [6]:
# Ask the model with NO course context.
# The answer is generic because the model doesn't know our course rules.
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Absolutely — if enrollment is still open, you can usually join now.

If you want, I can help you figure out the best next step:
- **If the course has already started:** ask whether there’s a **late enrollment** option or a way to catch up.
- **If it’s self-paced:** you can likely start immediately.
- **If there’s an application deadline:** you may need to submit it first.

If you’d like, I can also help you write a quick message to the course organizer asking if you can still enroll.


In [7]:
# Hand-written context: a few course FAQ entries.
# This is the "knowledge" we want the model to answer from.
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
# Build the final prompt: instructions + question + context.
# The "If the answer is not found ... say I don't know" rule keeps the model
# grounded in the context and reduces hallucination.
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
# Inspect the assembled prompt before sending it to the model.
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [10]:
# Now the answer is grounded in our context, not generic knowledge.
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [11]:
# Sketch of the full RAG pipeline we're building toward:
#   1) search()       -> retrieve documents relevant to the question
#   2) build_prompt()  -> combine question + retrieved docs into a prompt
#   3) llm()           -> generate the final answer
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

In [12]:
# Download the list of courses (each entry has the path to its FAQ file).
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

# courses_raw

In [13]:
# Flatten every course's FAQ into one big list of documents.
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    # Build the URL for this course's FAQ JSON
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() # stop early if the request fails
    course_data = course_response.json()

    documents.extend(course_data) # add this course's entries to the list

len(documents) # total number of FAQ documents collected

1350

In [14]:
# Peek at one document to see its structure
# (id, course, section, question, answer).
documents[1100]

{'id': 'ed90e0f589',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Bind for 0.0.0.0:9696 failed: port is already allocated',
 'answer': 'I was getting the following error when I rebuilt the Docker image, although the port was not allocated, and it was working fine.\n\nError message:\n\n```\nError response from daemon: driver failed programming external connectivity on endpoint beautiful_tharp (875be95c7027cebb853a62fc4463d46e23df99e0175be73641269c3d180f7796): Bind for 0.0.0.0:9696 failed: port is already allocated.\n```\n\n\n\nThe issue can be resolved by running the following command:\n\n```bash\ndocker kill $(docker ps -q)\n```\n\nFor more information, refer to the [GitHub issue on Docker for Windows](https://github.com/docker/for-win/issues/2722).'}

In [15]:
from minsearch import Index

# Build a small in-memory search index over the FAQ documents.
#   text_fields    -> fields searched with full-text / keyword relevance (TF-IDF style)
#   keyword_fields -> fields used for exact-match filtering (e.g. only one course)
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

# Feed the documents into the index so it can be searched later.
# Each dict in `documents` must contain the fields listed above.
index.fit(documents)

In [21]:
# Search again, but now weight some text fields differently using boost_dict.
#   boost_dict -> per-field multiplier on the relevance score
#     'question': 2.0 -> matches in the question field count twice as much
#     'section':  0.5 -> matches in the section field count half as much
#     (any text field not listed keeps its default weight of 1.0)
#   filter_dict -> still restrict to llm-zoomcamp documents only
#   num_results -> return the top 5 matches
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

# Inspect the returned list of matching documents (each is a dict).
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [23]:
def search(question, course='llm-zoomcamp'):
    # Field weights: question matches count double, section counts half.
    boost_dict = {'question': 2.0, 'section': 0.5}
    # Restrict results to the given course only.
    filter_dict = {'course': course}

    # Run the search and return the top 5 matching documents.
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [25]:
# Call our search() helper with the question.
# course defaults to 'llm-zoomcamp', so we don't need to pass it here.
# Returns the top 5 matching documents as a list of dicts.
search_results = search(question)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c